<a href="https://colab.research.google.com/github/AswinG2003/NLP-Parallel-Project/blob/main/NLP_Parallel_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import Libraries


In [2]:
# Install Required Packages (Run Once)


!pip install -q gensim wordcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.5 MB/s eta 0:00:00


In [5]:
# Download required NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Visualization settings
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP Libraries

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


# Machine Learning Utilities

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve
)


# Classical Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


# Word2Vec

import gensim.downloader as api
from gensim.models import Word2Vec


# Deep Learning (TensorFlow / Keras)
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Concatenate
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

#Load Dataset


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
df_twitter = pd.read_csv("/content/drive/MyDrive/DATA/twitter_training[1].csv")
df_twitter.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [12]:
# Define meaningful column names
column_names = ["Tweet_ID", "Entity", "Sentiment", "Tweet"]

# Load the dataset
df_twitter = pd.read_csv(
    "/content/drive/MyDrive/DATA/twitter_training[1].csv",
    names=column_names,
    header=None
)

# Display first five rows
df_twitter.head()

,Tweet_ID,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


#EDA


In [13]:
# Display basic information about the dataset

df_twitter.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Tweet_ID   74682 non-null  int64 
 1   Entity     74682 non-null  object
 2   Sentiment  74682 non-null  object
 3   Tweet      73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [14]:
print(f"Number of Rows    : {df_twitter.shape[0]}")
print(f"Number of Columns : {df_twitter.shape[1]}")

Number of Rows    : 74682
Number of Columns : 4


In [15]:
df_twitter.columns

Index(['Tweet_ID', 'Entity', 'Sentiment', 'Tweet'], dtype='object')

In [16]:
df_twitter.describe(include="all")

,Tweet_ID,Entity,Sentiment,Tweet
count,74682.000000,74682,74682,73996
unique,NaN,32,4,69491
top,NaN,Microsoft,Negative,"At the same time, despite the fact that there ..."
freq,NaN,2400,22542,172
mean,6432.586165,NaN,NaN,NaN
std,3740.427870,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN
25%,3195.000000,NaN,NaN,NaN
50%,6422.000000,NaN,NaN,NaN
75%,9601.000000,NaN,NaN,NaN


##Missing Value Analysis

In [17]:
# Check missing values

missing_values = df_twitter.isnull().sum()

missing_values

,0
Tweet_ID,0
Entity,0
Sentiment,0
Tweet,686


In [18]:
# Missing value percentage

missing_percentage = (df_twitter.isnull().sum() / len(df_twitter)) * 100

missing_df = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percentage.round(2)
})

missing_df

,Missing Values,Percentage (%)
Tweet_ID,0,0.00
Entity,0,0.00
Sentiment,0,0.00
Tweet,686,0.92


##

## Duplicate Value Analysis

In [19]:
# Check duplicate rows

duplicates = df_twitter.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 2700


#NLP Preprocessing

## Handling Missing Values


In [20]:
# Remove rows with missing tweets
df_twitter.dropna(subset=["Tweet"], inplace=True)

print("Dataset Shape after removing missing values:", df_twitter.shape)

Dataset Shape after removing missing values: (73996, 4)


##Handling Duplicates

In [21]:
# Remove duplicate records
df_twitter.drop_duplicates(inplace=True)

print("Dataset Shape after removing duplicates:", df_twitter.shape)

Dataset Shape after removing duplicates: (71656, 4)


##Lowercasing

In [22]:
# Convert tweets to lowercase
df_twitter["Tweet"] = df_twitter["Tweet"].str.lower()

df_twitter[["Tweet"]].head()

,Tweet
0,im getting on borderlands and i will murder yo...
1,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you ...
3,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...


## Punctuations

In [26]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_twitter["Tweet"] = df_twitter["Tweet"].apply(remove_punctuation)

df_twitter[["Tweet"]].head()

,Tweet
0,im getting on borderlands and i will murder yo...
1,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you all
3,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...


## Remove Numbers

In [28]:
import re

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

df_twitter["Tweet"] = df_twitter["Tweet"].apply(remove_numbers)

df_twitter[["Tweet"]].head()

,Tweet
0,im getting on borderlands and i will murder yo...
1,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you all
3,im coming on borderlands and i will murder you...
4,im getting on borderlands and i will murder y...


## Remove URL's

In [29]:
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

df_twitter["Tweet"] = df_twitter["Tweet"].apply(remove_urls)

df_twitter[["Tweet"]].head()

,Tweet
0,im getting on borderlands and i will murder yo...
1,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you all
3,im coming on borderlands and i will murder you...
4,im getting on borderlands and i will murder y...


## Remove StopWords

In [31]:
import nltk
nltk.download("punkt_tab")

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df_twitter["Tweet"] = df_twitter["Tweet"].apply(remove_stopwords)

df_twitter[["Tweet"]].head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,Tweet
0,im getting borderlands murder
1,coming borders kill
2,im getting borderlands kill
3,im coming borderlands murder
4,im getting borderlands murder


## Lemmatization

In [32]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

df_twitter["Tweet"] = df_twitter["Tweet"].apply(lemmatize_text)

df_twitter[["Tweet"]].head()

,Tweet
0,im getting borderland murder
1,coming border kill
2,im getting borderland kill
3,im coming borderland murder
4,im getting borderland murder


# Feature Engineering

## TF-IDF Vectorization

In [33]:
# Define input and target

X = df_twitter["Tweet"]
y = df_twitter["Sentiment"]

In [34]:
# Initialize TF-IDF Vectorizer

tfidf = TfidfVectorizer(max_features=5000)


In [35]:
# Transform text into TF-IDF features

X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Matrix Shape:", X_tfidf.shape)

TF-IDF Matrix Shape: (71656, 5000)


## Word2Vec Embeddings

In [36]:
# Tokenize each tweet into words
tokenized_tweets = df_twitter["Tweet"].apply(word_tokenize)

# Display the first five tokenized tweets
tokenized_tweets.head()

,Tweet
0,"[im, getting, borderland, murder]"
1,"[coming, border, kill]"
2,"[im, getting, borderland, kill]"
3,"[im, coming, borderland, murder]"
4,"[im, getting, borderland, murder]"


In [37]:
# Train the Word2Vec model
word2vec_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [38]:
# Generate document embeddings by averaging word vectors

X_word2vec = []

for tokens in tokenized_tweets:
    word_vectors = [
        word2vec_model.wv[word]
        for word in tokens
        if word in word2vec_model.wv
    ]

    if len(word_vectors) > 0:
        X_word2vec.append(np.mean(word_vectors, axis=0))
    else:
        X_word2vec.append(np.zeros(word2vec_model.vector_size))

X_word2vec = np.array(X_word2vec)

In [39]:
print("Word2Vec Shape:", X_word2vec.shape)

Word2Vec Shape: (71656, 100)


In [40]:
#The Word2Vec model converts each tweet into a dense numerical vector by averaging the embeddings of all words in the tweet. These embeddings capture semantic relationships between words and will be used for comparison with TF-IDF representations.

# Train-Test Split

In [41]:
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [42]:
X_train_w2v, X_test_w2v, _, _ = train_test_split(
    X_word2vec,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [43]:
print("TF-IDF Train :", X_train_tfidf.shape)
print("TF-IDF Test  :", X_test_tfidf.shape)

print("Word2Vec Train :", X_train_w2v.shape)
print("Word2Vec Test  :", X_test_w2v.shape)

TF-IDF Train : (57324, 5000)
TF-IDF Test  : (14332, 5000)
Word2Vec Train : (57324, 100)
Word2Vec Test  : (14332, 100)


# Model Building


## Logistic Regression

In [44]:
# Initialize Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [45]:
# Predict on the test set
y_pred_lr = lr_model.predict(X_test_tfidf)

In [46]:
# Calculate accuracy
lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.6801562936087078


In [47]:
# Classification Report
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

  Irrelevant       0.68      0.53      0.60      2507
    Negative       0.72      0.77      0.75      4340
     Neutral       0.63      0.64      0.63      3542
    Positive       0.68      0.71      0.69      3943

    accuracy                           0.68     14332
   macro avg       0.68      0.66      0.67     14332
weighted avg       0.68      0.68      0.68     14332



##### Inference: The Logistic Regression model achieved an overall accuracy of 68.02% on the test dataset. The model performed best in identifying Negative tweets, achieving the highest precision, recall, and F1-score. Performance on the Irrelevant class was comparatively lower, indicating that the model found it more difficult to distinguish irrelevant tweets from the other sentiment categories.

## SVM

In [49]:
# Initialize the Linear SVM model
svm_model = LinearSVC(random_state=42)

# Train the model
svm_model.fit(X_train_tfidf, y_train)

LinearSVC(random_state=42)

In [50]:
# Initialize the Linear SVM model
svm_model = LinearSVC(random_state=42)

# Train the model
svm_model.fit(X_train_tfidf, y_train)

LinearSVC(random_state=42)

In [51]:
# Predict the sentiments for the test data
y_pred_svm = svm_model.predict(X_test_tfidf)

In [52]:
# Calculate accuracy
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", svm_accuracy)

Linear SVM Accuracy: 0.7002511861568518


In [53]:
# Display classification report
print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

  Irrelevant       0.69      0.58      0.63      2507
    Negative       0.75      0.78      0.76      4340
     Neutral       0.65      0.67      0.66      3542
    Positive       0.70      0.72      0.71      3943

    accuracy                           0.70     14332
   macro avg       0.70      0.69      0.69     14332
weighted avg       0.70      0.70      0.70     14332



The Linear SVM model achieved an overall accuracy of 70.03%, outperforming Logistic Regression. It showed improved performance across most sentiment classes, particularly for the Negative and Positive categories, making it the better-performing classical machine learning model for this dataset.

# Deep Learning Model

#Model Evalutation Deep learning

# ROC Curve classical and Deep Learning

# Precision Recall Curve

# Final Model Comparison



# Conclusion